# Social Media Sentiment Analysis Pipeline

This notebook collects comments from YouTube and Reddit, cleans and
preprocesses the text, labels sentiment using both a rule-based method
(VADER) and a pre-trained transformer, optionally fine-tunes a
DistilBERT model, and produces a dashboard-ready export.

---

## Table of Contents

1. Setup and Configuration
2. Data Collection (YouTube and Reddit)
3. Text Preprocessing
4. Sentiment Labeling (VADER Auto-Label)
5. Train/Test Split
6. Fine-Tune a Transformer (Optional)
7. Classify with a Pre-Trained Model
8. Combine Sources and Export
9. Interactive Dashboard

---
## 1. Setup and Configuration

Install all required packages, import libraries, and define every
configurable parameter in one place. API keys are loaded from
environment variables for security. Copy `.env.example` to `.env` and
fill in your credentials before running.

In [ ]:
# ---------------------------------------------------------------------------
# Install dependencies (run once)
# ---------------------------------------------------------------------------
!pip install -q     transformers datasets evaluate scikit-learn     vaderSentiment langdetect torch     praw google-api-python-client python-dotenv     streamlit plotly wordcloud pyngrok

In [ ]:
# ---------------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------------
import csv
import json
import logging
import os
import re
import time
from collections import Counter
from html import unescape
from pathlib import Path
from typing import Literal, Optional

import matplotlib.pyplot as plt
import nltk
import pandas as pd
from dotenv import load_dotenv
from langdetect import detect, LangDetectException
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# HuggingFace
from datasets import load_dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    pipeline as hf_pipeline,
)
import evaluate

# APIs
from googleapiclient.discovery import build
import praw

# NLTK resources
for _res in ("stopwords", "wordnet"):
    nltk.download(_res, quiet=True)

load_dotenv()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("sentiment")

In [ ]:
# ---------------------------------------------------------------------------
# Configuration -- edit these values to match your project
# ---------------------------------------------------------------------------

# API credentials (set in .env or replace the fallback strings)
YOUTUBE_API_KEY: str   = os.getenv("YOUTUBE_API_KEY", "YOUR_YOUTUBE_API_KEY_HERE")
REDDIT_CLIENT_ID: str  = os.getenv("REDDIT_CLIENT_ID", "YOUR_REDDIT_CLIENT_ID_HERE")
REDDIT_CLIENT_SECRET: str = os.getenv("REDDIT_CLIENT_SECRET", "YOUR_REDDIT_CLIENT_SECRET_HERE")
REDDIT_USER_AGENT: str = os.getenv(
    "REDDIT_USER_AGENT",
    "sentiment-analysis-pipeline v1.0 (by u/your_username)",
)

# YouTube collection
YOUTUBE_SEARCH_KEYWORDS: list[str] = ["Tesla", "Tesla Cybertruck"]
YOUTUBE_VIDEO_IDS: list[str]       = []   # add specific video IDs here if needed
YT_MAX_VIDEOS_PER_KEYWORD: int     = 10
YT_MAX_COMMENTS_PER_VIDEO: int     = 200

# Reddit collection
REDDIT_SEARCH_QUERY: str    = "cybertruck"
REDDIT_MAX_COMMENTS: int    = 1500
REDDIT_SUBMISSION_LIMIT: int = 100

# Pre-trained model for inference (no training required)
PRETRAINED_MODEL: str = "distilbert-base-uncased-finetuned-sst-2-english"

# Fine-tuning settings
FINETUNE_BASE_MODEL: str  = "distilbert-base-uncased"
FINETUNE_MAX_LENGTH: int  = 128
FINETUNE_EPOCHS: int      = 3
FINETUNE_LR: float        = 2e-5
FINETUNE_BATCH_SIZE: int  = 16
FINETUNE_WEIGHT_DECAY: float = 0.01
FINETUNE_TEST_SIZE: float = 0.1

# VADER thresholds
VADER_POS_THRESHOLD: float  =  0.05
VADER_NEG_THRESHOLD: float  = -0.05

# Output directory
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

print("Configuration loaded.")

---
## 2. Data Collection

### 2.1 YouTube

The YouTube collector searches for videos by keyword, then paginates
through each video's comment threads. Each comment record includes the
source platform, video ID, author, text, timestamp, and like count.

In [ ]:
# ---------------------------------------------------------------------------
# YouTube helpers
# ---------------------------------------------------------------------------

def yt_search_videos(query: str, max_results: int = 50) -> list[str]:
    """Search YouTube and return a list of video IDs."""
    youtube = build("youtube", "v3", developerKey=YOUTUBE_API_KEY)
    request = youtube.search().list(
        q=query, part="id", type="video", maxResults=min(max_results, 50),
    )
    response = request.execute()
    ids = [item["id"]["videoId"] for item in response.get("items", [])]
    logger.info("Search '%s' returned %d videos.", query, len(ids))
    return ids


def yt_fetch_comments(video_id: str, max_comments: int = 200) -> list[dict]:
    """Fetch top-level comments for a single YouTube video."""
    youtube = build("youtube", "v3", developerKey=YOUTUBE_API_KEY)
    comments: list[dict] = []

    request = youtube.commentThreads().list(
        part="snippet", videoId=video_id,
        maxResults=100, textFormat="plainText",
    )

    while request and len(comments) < max_comments:
        response = request.execute()
        for item in response.get("items", []):
            s = item["snippet"]["topLevelComment"]["snippet"]
            comments.append({
                "source":           "youtube",
                "video_id":         video_id,
                "comment_id":       item["id"],
                "text":             s.get("textDisplay", ""),
                "author":           s.get("authorDisplayName", ""),
                "published_at":     s.get("publishedAt", ""),
                "like_count":       s.get("likeCount", 0),
                "upvotes":          None,
                "replies_count":    None,
                "engagement_score": s.get("likeCount", 0),
            })
        request = youtube.commentThreads().list_next(request, response)
        time.sleep(0.1)

    logger.info("Video %s: fetched %d comments.", video_id, len(comments))
    return comments

In [ ]:
# ---------------------------------------------------------------------------
# Collect YouTube comments
# ---------------------------------------------------------------------------
yt_records: list[dict] = []
seen_videos: set[str] = set()

for kw in YOUTUBE_SEARCH_KEYWORDS:
    video_ids = yt_search_videos(kw, max_results=YT_MAX_VIDEOS_PER_KEYWORD)
    for vid in video_ids:
        if vid not in seen_videos:
            seen_videos.add(vid)
            yt_records.extend(yt_fetch_comments(vid, YT_MAX_COMMENTS_PER_VIDEO))

# Add any explicitly listed video IDs
for vid in YOUTUBE_VIDEO_IDS:
    if vid not in seen_videos:
        seen_videos.add(vid)
        yt_records.extend(yt_fetch_comments(vid, YT_MAX_COMMENTS_PER_VIDEO))

df_youtube_raw = pd.DataFrame(yt_records)
df_youtube_raw.to_parquet(DATA_DIR / "youtube_raw.parquet", index=False)
print(f"YouTube: collected {len(df_youtube_raw)} comments.")
df_youtube_raw.head()

### 2.2 Reddit

The Reddit collector uses PRAW to search all subreddits for a query
string, then flattens every comment tree into individual records.
Engagement score is computed as upvotes plus reply count.

In [ ]:
# ---------------------------------------------------------------------------
# Reddit helpers
# ---------------------------------------------------------------------------

def reddit_collect(
    query: str,
    max_comments: int = 1500,
    submission_limit: int = 100,
) -> pd.DataFrame:
    """Search Reddit and collect comments matching a query."""
    reddit = praw.Reddit(
        client_id=REDDIT_CLIENT_ID,
        client_secret=REDDIT_CLIENT_SECRET,
        user_agent=REDDIT_USER_AGENT,
    )
    comments: list[dict] = []

    for submission in reddit.subreddit("all").search(query, limit=submission_limit):
        submission.comments.replace_more(limit=0)
        for comment in submission.comments.list():
            if len(comments) >= max_comments:
                break
            if not comment.author or comment.author.name == "[deleted]":
                continue
            reply_count = len(comment.replies)
            comments.append({
                "source":           "reddit",
                "video_id":         None,
                "subreddit":        submission.subreddit.display_name,
                "submission_id":    submission.id,
                "comment_id":       comment.id,
                "author":           comment.author.name,
                "text":             comment.body,
                "published_at":     pd.to_datetime(comment.created_utc, unit="s"),
                "like_count":       None,
                "upvotes":          comment.score,
                "replies_count":    reply_count,
                "engagement_score": comment.score + reply_count,
            })
        if len(comments) >= max_comments:
            break
        time.sleep(1)

    return pd.DataFrame(comments)

In [ ]:
# ---------------------------------------------------------------------------
# Collect Reddit comments
# ---------------------------------------------------------------------------
df_reddit_raw = reddit_collect(
    query=REDDIT_SEARCH_QUERY,
    max_comments=REDDIT_MAX_COMMENTS,
    submission_limit=REDDIT_SUBMISSION_LIMIT,
)
df_reddit_raw.to_csv(DATA_DIR / "reddit_raw.csv", index=False)
print(f"Reddit: collected {len(df_reddit_raw)} comments.")
df_reddit_raw.head()

---
## 3. Text Preprocessing

Every comment passes through a five-step cleaning pipeline:

1. Strip HTML tags and decode HTML entities.
2. Lowercase and collapse whitespace.
3. Remove stopwords while preserving negation words (not, no, never, etc.)
   that carry sentiment meaning.
4. Lemmatize tokens to their dictionary form.
5. Detect language and keep only English comments.

In [ ]:
# ---------------------------------------------------------------------------
# Preprocessing functions
# ---------------------------------------------------------------------------
_lemmatizer = WordNetLemmatizer()

_stop_words: set[str] = set(stopwords.words("english"))
_NEGATION_WORDS = {"not", "no", "nor", "neither", "never", "nobody", "nothing", "nowhere"}
_stop_words -= _NEGATION_WORDS


def strip_html(text: str) -> str:
    """Remove HTML tags and decode entities."""
    text = unescape(text)
    return re.sub(r"<[^>]+>", " ", text)


def normalize(text: str) -> str:
    """Lowercase and collapse whitespace."""
    return re.sub(r"\s+", " ", text.lower()).strip()


def remove_stopwords(text: str) -> str:
    """Remove common stopwords, keeping negation."""
    return " ".join(t for t in text.split() if t not in _stop_words)


def lemmatize(text: str) -> str:
    """Reduce each token to its dictionary form."""
    return " ".join(_lemmatizer.lemmatize(t) for t in text.split())


def detect_language(text: str) -> str:
    """Return ISO-639-1 language code, or 'unk' on failure."""
    if not re.search(r"[a-zA-Z]", text):
        return "unk"
    try:
        return detect(text)
    except LangDetectException:
        return "unk"


def preprocess(text: Optional[str]) -> str:
    """Run the full cleaning pipeline on a single text string."""
    if text is None:
        return ""
    text = str(text)
    text = strip_html(text)
    text = normalize(text)
    text = remove_stopwords(text)
    text = lemmatize(text)
    return text if text.strip() else ""

In [ ]:
# ---------------------------------------------------------------------------
# Apply preprocessing to YouTube data
# ---------------------------------------------------------------------------
df_youtube = df_youtube_raw.copy()
df_youtube["text_clean"] = df_youtube["text"].apply(preprocess)
df_youtube["lang"] = df_youtube["text_clean"].apply(detect_language)
df_youtube = df_youtube[df_youtube["lang"] == "en"].copy()

df_youtube.to_csv(DATA_DIR / "youtube_clean.csv", index=False)
print(f"After cleaning and language filter: {len(df_youtube)} English comments.")
df_youtube.head()

---
## 4. Sentiment Labeling with VADER

VADER (Valence Aware Dictionary and sEntiment Reasoner) is a rule-based
sentiment analyzer tuned for social media text. It produces a compound
score between -1 and +1. Comments are labeled as:

- positive if compound >= 0.05
- negative if compound <= -0.05
- neutral otherwise

These auto-labels serve as ground truth for the optional fine-tuning
stage below.

In [ ]:
# ---------------------------------------------------------------------------
# VADER labeling
# ---------------------------------------------------------------------------
_vader = SentimentIntensityAnalyzer()


def label_vader(text: str) -> Literal["positive", "negative", "neutral"]:
    """Classify a single text using VADER compound score."""
    compound = _vader.polarity_scores(str(text))["compound"]
    if compound >= VADER_POS_THRESHOLD:
        return "positive"
    if compound <= VADER_NEG_THRESHOLD:
        return "negative"
    return "neutral"


df_youtube["label"] = df_youtube["text_clean"].apply(label_vader)
df_youtube.to_csv(DATA_DIR / "youtube_labeled.csv", index=False)

print("Label distribution:")
print(df_youtube["label"].value_counts())

---
## 5. Train / Test Split

A stratified 90/10 split preserves the label distribution in both
partitions. The random seed is fixed for reproducibility.

In [ ]:
# ---------------------------------------------------------------------------
# Stratified split
# ---------------------------------------------------------------------------
df_train, df_test = train_test_split(
    df_youtube,
    test_size=FINETUNE_TEST_SIZE,
    stratify=df_youtube["label"],
    random_state=42,
)

df_train.to_csv(DATA_DIR / "train.csv", index=False)
df_test.to_csv(DATA_DIR / "test.csv", index=False)
print(f"Train: {len(df_train)} rows  |  Test: {len(df_test)} rows")

---
## 6. Fine-Tune a DistilBERT Model (Optional)

This section fine-tunes `distilbert-base-uncased` on the VADER-labeled
data using the HuggingFace Trainer API. Skip this section if you only
want to use the pre-trained classifier in Section 7.

The training loop evaluates accuracy at the end of every epoch and
saves the best checkpoint.

In [ ]:
# ---------------------------------------------------------------------------
# Fine-tuning
# ---------------------------------------------------------------------------

# 1. Load data
dataset = load_dataset("csv", data_files={
    "train": str(DATA_DIR / "train.csv"),
    "test":  str(DATA_DIR / "test.csv"),
})

# 2. Map string labels to integers
label_names = sorted(dataset["train"].unique("label"))
label2id = {name: idx for idx, name in enumerate(label_names)}
id2label = {idx: name for name, idx in label2id.items()}
print("Label mapping:", label2id)

def encode_labels(batch):
    batch["labels"] = [label2id[x] for x in batch["label"]]
    return batch

dataset = dataset.map(encode_labels, batched=True)

# 3. Tokenize
tokenizer = AutoTokenizer.from_pretrained(FINETUNE_BASE_MODEL)

def tokenize(batch):
    return tokenizer(
        batch["text_clean"],
        padding="max_length",
        truncation=True,
        max_length=FINETUNE_MAX_LENGTH,
    )

dataset = dataset.map(tokenize, batched=True)
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# 4. Model
model = AutoModelForSequenceClassification.from_pretrained(
    FINETUNE_BASE_MODEL,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id,
)

# 5. Training arguments
training_args = TrainingArguments(
    output_dir="results",
    learning_rate=FINETUNE_LR,
    per_device_train_batch_size=FINETUNE_BATCH_SIZE,
    per_device_eval_batch_size=FINETUNE_BATCH_SIZE,
    num_train_epochs=FINETUNE_EPOCHS,
    weight_decay=FINETUNE_WEIGHT_DECAY,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=50,
)

# 6. Metrics
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return accuracy_metric.compute(predictions=preds, references=labels)

# 7. Train
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

# 8. Evaluate and save
results = trainer.evaluate()
print("Evaluation results:", results)

MODEL_DIR = Path("model/sentiment")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(MODEL_DIR))
tokenizer.save_pretrained(str(MODEL_DIR))
print(f"Model saved to {MODEL_DIR}.")

---
## 7. Classify with a Pre-Trained Model

This section uses a HuggingFace sentiment-analysis pipeline to classify
both YouTube and Reddit comments without any fine-tuning. The default
model is `distilbert-base-uncased-finetuned-sst-2-english` (binary
positive/negative). You can swap it for a three-class model like
`cardiffnlp/twitter-roberta-base-sentiment` by changing
`PRETRAINED_MODEL` in the configuration section.

In [ ]:
# ---------------------------------------------------------------------------
# Pre-trained classification
# ---------------------------------------------------------------------------
classifier = hf_pipeline(
    "sentiment-analysis",
    model=PRETRAINED_MODEL,
    truncation=True,
    padding=True,
)


def classify_df(df: pd.DataFrame, text_col: str = "text") -> pd.DataFrame:
    """Run the pre-trained classifier over a DataFrame."""
    df = df.copy()
    texts = df[text_col].fillna("").tolist()
    results = classifier(texts, batch_size=64)
    df["sentiment"]  = [r["label"].lower() for r in results]
    df["confidence"] = [round(r["score"], 4) for r in results]
    return df


# Classify YouTube
df_yt_classified = classify_df(df_youtube_raw, text_col="text")
df_yt_classified.to_csv(DATA_DIR / "youtube_classified.csv", index=False)
print(f"YouTube classified: {len(df_yt_classified)} rows")
print(df_yt_classified["sentiment"].value_counts())

In [ ]:
# Classify Reddit
df_rd_classified = classify_df(df_reddit_raw, text_col="text")
df_rd_classified.to_csv(DATA_DIR / "reddit_classified.csv", index=False)
print(f"Reddit classified: {len(df_rd_classified)} rows")
print(df_rd_classified["sentiment"].value_counts())

---
## 8. Combine Sources and Export

Both classified DataFrames are concatenated into a single dataset.
Timestamps are normalized (handling both ISO-8601 strings from YouTube
and Unix epochs from Reddit). Text columns are sanitized to remove
newlines and problematic characters. The final CSV uses UTF-8 with BOM
encoding and quotes every field so that dashboard tools like Looker
Studio and Excel parse it without errors.

In [ ]:
# ---------------------------------------------------------------------------
# Combine and normalize
# ---------------------------------------------------------------------------

def parse_timestamp(value) -> pd.Timestamp:
    """Parse ISO-8601 or Unix epoch timestamps to UTC."""
    try:
        return pd.to_datetime(value, utc=True)
    except (ValueError, TypeError):
        pass
    try:
        return pd.to_datetime(float(value), unit="s", utc=True)
    except (ValueError, TypeError):
        return pd.NaT


def sanitize_text_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Remove newlines and problematic quotes from text columns."""
    df = df.copy()
    for col in df.select_dtypes(include="object").columns:
        df[col] = (
            df[col].astype(str)
            .str.replace("\n", " ", regex=False)
            .str.replace("\r", " ", regex=False)
            .str.replace('"', "'", regex=False)
        )
    return df


# Concatenate
df_combined = pd.concat([df_yt_classified, df_rd_classified], ignore_index=True)
df_combined["published_at"] = df_combined["published_at"].apply(parse_timestamp)

missing = df_combined["published_at"].isna().sum()
print(f"Combined: {len(df_combined)} rows  |  Missing timestamps: {missing}")

In [ ]:
# ---------------------------------------------------------------------------
# Export dashboard-ready CSV
# ---------------------------------------------------------------------------
df_export = df_combined.copy()

# Format timestamps as clean strings
df_export["published_at"] = pd.to_datetime(
    df_export["published_at"], errors="coerce", utc=True,
).dt.strftime("%Y-%m-%d %H:%M:%S")

df_export = sanitize_text_columns(df_export)

export_path = DATA_DIR / "dashboard_export.csv"
df_export.to_csv(
    export_path,
    index=False,
    encoding="utf-8-sig",
    quoting=csv.QUOTE_ALL,
)

print(f"Exported {len(df_export)} rows x {len(df_export.columns)} columns to {export_path}")
df_export.head()

---
## 9. Interactive Dashboard (Streamlit)

The cell below writes a standalone Streamlit app to `dashboard.py` and
launches it through an ngrok tunnel so you can access it from any
browser. The dashboard includes:

- KPI cards (total comments, positive/negative percentage, avg engagement)
- Sentiment pie chart
- Comment volume by platform
- Top 10 authors by engagement
- Word clouds for positive and negative comments
- Keyword bubble chart (word frequency vs. average engagement)
- Table of the 20 most engaging comments

To run it locally outside this notebook:

```
streamlit run dashboard.py
```

In [ ]:
%%writefile dashboard.py
import re
from collections import Counter

import matplotlib.pyplot as plt
import nltk
import pandas as pd
import plotly.express as px
import streamlit as st
from wordcloud import WordCloud

nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

_STOP = set(stopwords.words("english")) | {
    "amp", "https", "http", "www", "video", "com", "like", "get", "one", "us", "also",
}


@st.cache_data
def load_data(path):
    df = pd.read_csv(path)
    df["published_at"] = pd.to_datetime(df["published_at"], errors="coerce")
    df["sentiment"] = df["sentiment"].astype(str).str.lower().str.strip()
    df["source"] = (
        df["source"].astype(str).str.strip().str.lower()
        .replace({"yt": "youtube", "reddit.com": "reddit"})
    )
    return df


def extract_words(texts):
    blob = " ".join(texts.dropna())
    blob = re.sub(r"http\S+|www\S+|https\S+", "", blob.lower())
    words = re.findall(r"\b[a-z]{3,}\b", blob)
    return [w for w in words if w not in _STOP]


def main():
    st.set_page_config(page_title="Sentiment Dashboard", layout="wide")
    st.title("Social Media Sentiment Dashboard")

    st.sidebar.title("Filters")
    data_path = st.sidebar.text_input("CSV path", value="data/dashboard_export.csv")
    df = load_data(data_path)

    platform = st.sidebar.radio("Platform", ["All", "youtube", "reddit"])
    if platform != "All":
        df = df[df["source"] == platform]

    if df.empty:
        st.warning(f"No data for '{platform}'.")
        return

    total = len(df)
    pos = (df["sentiment"] == "positive").sum()
    neg = (df["sentiment"] == "negative").sum()
    eng = df["engagement_score"].mean() if "engagement_score" in df.columns else 0

    c1, c2, c3, c4 = st.columns(4)
    c1.metric("Total Comments", f"{total:,}")
    c2.metric("Positive", f"{pos/total*100:.1f}%")
    c3.metric("Negative", f"{neg/total*100:.1f}%")
    c4.metric("Avg Engagement", f"{eng:.1f}")

    st.subheader("Sentiment Breakdown")
    counts = df["sentiment"].value_counts().reset_index()
    counts.columns = ["sentiment", "count"]
    fig = px.pie(counts, names="sentiment", values="count", color="sentiment",
                 color_discrete_map={"positive": "#2ecc71", "negative": "#e74c3c", "neutral": "#95a5a6"})
    st.plotly_chart(fig, use_container_width=True)

    st.subheader("Volume by Platform")
    vol = df.groupby("source").size().reset_index(name="count")
    st.plotly_chart(px.bar(vol, x="source", y="count", color="source"), use_container_width=True)

    if "engagement_score" in df.columns:
        st.subheader("Top 10 Authors by Engagement")
        top_auth = df.groupby("author")["engagement_score"].sum().nlargest(10).reset_index()
        st.plotly_chart(px.bar(top_auth, x="author", y="engagement_score"), use_container_width=True)

    st.subheader("Word Clouds")
    col_a, col_b = st.columns(2)
    for col, sent, title in [(col_a, "positive", "Positive"), (col_b, "negative", "Negative")]:
        words = extract_words(df[df["sentiment"] == sent]["text"])
        wc = WordCloud(width=600, height=400, background_color="white").generate(" ".join(words) or "empty")
        fig_wc, ax = plt.subplots(figsize=(6, 4))
        ax.imshow(wc, interpolation="bilinear"); ax.axis("off"); ax.set_title(title)
        col.pyplot(fig_wc)

    st.subheader("Top Keywords by Engagement")
    kw_rows = []
    for sent in ("positive", "negative"):
        words = extract_words(df[df["sentiment"] == sent]["text"])
        for word, count in Counter(words).most_common(20):
            avg = df.loc[df["text"].str.contains(word, na=False), "engagement_score"].mean()
            kw_rows.append({"word": word, "count": count, "avg_engagement": avg, "sentiment": sent})
    kw_df = pd.DataFrame(kw_rows)
    if not kw_df.empty:
        st.plotly_chart(px.scatter(kw_df, x="word", y="avg_engagement", size="count", color="sentiment",
                                   color_discrete_map={"positive": "#2ecc71", "negative": "#e74c3c"}),
                        use_container_width=True)

    st.subheader("Top 20 Most Engaging Comments")
    if "engagement_score" in df.columns:
        st.dataframe(df.nlargest(20, "engagement_score")[["text", "sentiment", "engagement_score"]],
                      use_container_width=True)


if __name__ == "__main__":
    main()

In [ ]:
# ---------------------------------------------------------------------------
# Launch dashboard (Colab / cloud environment)
# ---------------------------------------------------------------------------
# Uncomment the lines below to start Streamlit with an ngrok tunnel.

# !nohup streamlit run dashboard.py --server.port 8501 &>/dev/null &
# !ngrok authtoken YOUR_NGROK_TOKEN
# from pyngrok import ngrok
# public_url = ngrok.connect(8501)
# print("Dashboard URL:", public_url)

---

## Summary

This notebook provides a complete, end-to-end sentiment analysis
pipeline in a single file. The key stages are:

1. **Collect** comments from YouTube (API v3) and Reddit (PRAW).
2. **Preprocess** text by removing HTML, normalizing, filtering stopwords
   (while keeping negation), lemmatizing, and filtering to English only.
3. **Label** with VADER for auto-generating training data.
4. **Fine-tune** a DistilBERT model on the labeled data (optional).
5. **Classify** all comments with a pre-trained HuggingFace pipeline.
6. **Export** a clean, dashboard-ready CSV with proper encoding and quoting.
7. **Visualize** with an interactive Streamlit dashboard.

All configuration is centralized at the top of the notebook. API keys
should be stored in a `.env` file and never hardcoded.